In [2]:
"""
merge_patents.py
----------------
Lee todos los archivos XLS/XLSX de una carpeta de entrada,
extrae las columnas relevantes y genera un único archivo Excel consolidado.

Uso:
    python merge_patents.py [carpeta_entrada] [archivo_salida]

Defaults:
    carpeta_entrada  = ./input      (carpeta donde están los XLS)
    archivo_salida   = ./merged_patents.xlsx
"""

import sys
import glob
import os
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ── Configuración ────────────────────────────────────────────────────────────
INPUT_FOLDER = r"C:\Users\david\Downloads\Patentes_OMPI"
OUTPUT_FILE  = r"C:\Users\david\Downloads\Patentes_OMPI\merged_patents.xlsx"

# Columnas canónicas de salida
OUTPUT_COLUMNS = [
    "Application_Id",
    "Application_Number",
    "Application_Date",
    "Country",
    "Title",
    "IPC",
]

# Mapeo flexible: variantes posibles de cada encabezado → nombre canónico
COLUMN_MAP = {
    "application id":     "Application_Id",
    "applicationid":      "Application_Id",
    "app id":             "Application_Id",
    "appid":              "Application_Id",

    "application number": "Application_Number",
    "applicationnumber":  "Application_Number",
    "app number":         "Application_Number",
    "appnumber":          "Application_Number",

    "application date":   "Application_Date",
    "applicationdate":    "Application_Date",
    "app date":           "Application_Date",
    "appdate":            "Application_Date",
    "filing date":        "Application_Date",

    "country":            "Country",

    "title":              "Title",

    "i p c":              "IPC",
    "ipc":                "IPC",
    "i.p.c":              "IPC",
    "i.p.c.":             "IPC",
    "ipc code":           "IPC",
}

# ── Funciones ─────────────────────────────────────────────────────────────────

def normalize_col(name: str) -> str:
    return str(name).strip().lower().replace("  ", " ")


def find_header_row(raw: pd.DataFrame) -> int:
    """Busca la fila que contiene los encabezados de datos."""
    for i, row in raw.iterrows():
        values = [normalize_col(v) for v in row if pd.notna(v)]
        # Si al menos 3 valores coinciden con columnas conocidas → es la cabecera
        matches = sum(1 for v in values if v in COLUMN_MAP)
        if matches >= 3:
            return i
    return 0


def read_file(path: str) -> pd.DataFrame | None:
    ext = os.path.splitext(path)[1].lower()
    engine = "xlrd" if ext == ".xls" else "openpyxl"

    try:
        raw = pd.read_excel(path, header=None, engine=engine)
    except Exception as e:
        print(f"  ⚠  No se pudo leer {os.path.basename(path)}: {e}")
        return None

    header_row = find_header_row(raw)

    # Re-leer usando esa fila como cabecera
    df = pd.read_excel(path, header=header_row, engine=engine)

    # Renombrar columnas al estándar
    rename = {}
    for col in df.columns:
        key = normalize_col(col)
        if key in COLUMN_MAP:
            rename[col] = COLUMN_MAP[key]
    df.rename(columns=rename, inplace=True)

    # Conservar solo las columnas que existen en el output
    present = [c for c in OUTPUT_COLUMNS if c in df.columns]
    missing = [c for c in OUTPUT_COLUMNS if c not in df.columns]
    if missing:
        print(f"  ℹ  {os.path.basename(path)}: columnas no encontradas → {missing}")

    df = df[present].copy()

    # Agregar columnas faltantes como NaN
    for c in missing:
        df[c] = pd.NA

    df = df[OUTPUT_COLUMNS]

    # Eliminar filas completamente vacías
    df.dropna(how="all", inplace=True)

    # Agregar columna de origen
    df.insert(0, "Source_File", os.path.basename(path))

    return df


def style_workbook(path: str):
    wb = load_workbook(path)
    ws = wb.active

    header_fill  = PatternFill("solid", fgColor="2F5496")
    header_font  = Font(name="Arial", bold=True, color="FFFFFF", size=10)
    cell_font    = Font(name="Arial", size=10)
    center       = Alignment(horizontal="center", vertical="center", wrap_text=True)
    left         = Alignment(horizontal="left",   vertical="center", wrap_text=True)
    thin_side    = Side(style="thin", color="BFBFBF")
    thin_border  = Border(left=thin_side, right=thin_side, top=thin_side, bottom=thin_side)

    col_widths = {
        "A": 28,   # Source_File
        "B": 18,   # Application_Id
        "C": 22,   # Application_Number
        "D": 18,   # Application_Date
        "E": 10,   # Country
        "F": 55,   # Title
        "G": 35,   # IPC
    }

    for col_letter, width in col_widths.items():
        ws.column_dimensions[col_letter].width = width

    ws.row_dimensions[1].height = 30

    for row_idx, row in enumerate(ws.iter_rows(), start=1):
        for cell in row:
            cell.border = thin_border
            if row_idx == 1:
                cell.fill   = header_fill
                cell.font   = header_font
                cell.alignment = center
            else:
                cell.font      = cell_font
                col = get_column_letter(cell.column)
                cell.alignment = center if col in ("D", "E") else left
                if row_idx % 2 == 0:
                    cell.fill = PatternFill("solid", fgColor="EEF2FA")

    ws.freeze_panes = "B2"
    wb.save(path)


# ── Main ──────────────────────────────────────────────────────────────────────

def main():
    patterns = [
        os.path.join(INPUT_FOLDER, "*.xls"),
        os.path.join(INPUT_FOLDER, "*.xlsx"),
        os.path.join(INPUT_FOLDER, "*.xlsm"),
    ]
    files = sorted(set(f for p in patterns for f in glob.glob(p)))

    if not files:
        print(f"No se encontraron archivos XLS/XLSX en: {INPUT_FOLDER}")
        sys.exit(1)

    print(f"Archivos encontrados: {len(files)}\n")

    frames = []
    for path in files:
        print(f"Procesando: {os.path.basename(path)}")
        df = read_file(path)
        if df is not None and not df.empty:
            frames.append(df)
            print(f"  ✓  {len(df)} registros")

    if not frames:
        print("No se extrajo ningún dato.")
        sys.exit(1)

    merged = pd.concat(frames, ignore_index=True)

    # Formatear fecha consistentemente si es posible
    merged["Application_Date"] = pd.to_datetime(
        merged["Application_Date"], dayfirst=True, errors="coerce"
    ).dt.strftime("%d.%m.%Y").fillna(merged["Application_Date"].astype(str))

    os.makedirs(os.path.dirname(os.path.abspath(OUTPUT_FILE)), exist_ok=True)
    merged.to_excel(OUTPUT_FILE, index=False)

    style_workbook(OUTPUT_FILE)

    print(f"\n{'─'*50}")
    print(f"✅ Merge completado")
    print(f"   Archivos procesados : {len(frames)}")
    print(f"   Total de registros  : {len(merged)}")
    print(f"   Archivo de salida   : {OUTPUT_FILE}")


if __name__ == "__main__":
    main()


Archivos encontrados: 21

Procesando: resultList (10).xls
  ✓  4196 registros
Procesando: resultList (11).xls
  ✓  1772 registros
Procesando: resultList (12).xls
  ✓  4063 registros
Procesando: resultList (13).xls
  ✓  3365 registros
Procesando: resultList (14).xls
  ✓  4388 registros
Procesando: resultList (15).xls
  ✓  6085 registros
Procesando: resultList (16).xls
  ✓  6085 registros
Procesando: resultList (17).xls
  ✓  3723 registros
Procesando: resultList (18).xls
  ✓  4564 registros
Procesando: resultList (19).xls
  ✓  4134 registros
Procesando: resultList (20).xls
  ✓  4738 registros
Procesando: resultList (21).xls
  ✓  3995 registros
Procesando: resultList (22).xls
  ✓  2119 registros
Procesando: resultList (23).xls
  ✓  888 registros
Procesando: resultList (3).xls
  ✓  2544 registros
Procesando: resultList (4).xls
  ✓  2845 registros
Procesando: resultList (5).xls
  ✓  2190 registros
Procesando: resultList (6).xls
  ✓  1707 registros
Procesando: resultList (7).xls
  ✓  1736 re